# Gradle Agent - Interactive Notebook

このnotebookでは、gradle_agentをインタラクティブに実行できます。

## 機能
- Gradleプロジェクトの解析
- モジュール構成の特定
- 依存関係の抽出
- 技術スタックの分析
- 解析結果をYAML形式で保存

In [ ]:
# セットアップ
import sys
import os
import yaml
sys.path.append('..')

from src.agents.gradle_agent import gradle_agent
from src.models.state import OverallState
from langchain_core.runnables import RunnableConfig
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# products.yamlをプロジェクトルートから読み込む
products_yaml_path = "../products.yaml"

if os.path.exists(products_yaml_path):
    with open(products_yaml_path, 'r', encoding='utf-8') as f:
        products_config = yaml.safe_load(f)
    print(f"✅ products.yaml を読み込みました")
    print(f"リポジトリ数: {len(products_config.get('repositories', []))}")
    for repo in products_config.get('repositories', []):
        print(f"  - {repo.get('id')}: {repo.get('name')} ({repo.get('path')})")
else:
    print(f"⚠️ products.yaml が見つかりません: {products_yaml_path}")
    products_config = {"repositories": [], "products": []}

# 初期状態（products.yamlから読み込んだ設定を使用）
state: OverallState = {
    "products_config": products_config,
    "results": {}
}

## 使い方1: products.yamlの全リポジトリを解析

products.yamlに定義されたすべてのGradleプロジェクトを自動解析します。

In [ ]:
# products.yamlの全リポジトリを解析
result = gradle_agent(state)
print("\n=== 解析完了! ===")
print("解析結果は metadata/ ディレクトリに保存されました。")

## 使い方2: 特定のリポジトリのみを解析

products.yamlを使わず、カスタムのリポジトリを指定する場合

In [ ]:
# カスタムのリポジトリを指定
REPO_PATH = "workspace/your-gradle-project"  # 解析したいGradleプロジェクトのパス
REPO_ID = "gradle-project"
REPO_NAME = "My Gradle Project"

# カスタム状態を作成
custom_state: OverallState = {
    "products_config": {
        "repositories": [
            {
                "id": REPO_ID,
                "name": REPO_NAME,
                "path": REPO_PATH
            }
        ]
    },
    "results": {}
}

# エージェントを実行
result = gradle_agent(custom_state)
print("\n=== 解析完了! ===")

## 使い方3: 複数のリポジトリを一度に解析

In [ ]:
# 複数のGradleプロジェクトを一度に解析
multi_state: OverallState = {
    "products_config": {
        "repositories": [
            {"id": "project1", "name": "Project 1", "path": "workspace/project1"},
            {"id": "project2", "name": "Project 2", "path": "workspace/project2"},
        ]
    },
    "results": {}
}

result = gradle_agent(multi_state)
print("\n=== 全プロジェクトの解析完了! ===")

## 解析結果の確認

In [ ]:
# 保存された解析結果を読み込んで表示
import yaml
import os

# メタデータディレクトリを確認
metadata_dir = "../metadata"
if os.path.exists(metadata_dir):
    repos = [d for d in os.listdir(metadata_dir) if os.path.isdir(os.path.join(metadata_dir, d))]
    print(f"解析済みリポジトリ: {repos}")
    
    # 最初のリポジトリの解析結果を表示
    if repos:
        analysis_file = os.path.join(metadata_dir, repos[0], "gradle_analysis.yaml")
        if os.path.exists(analysis_file):
            with open(analysis_file, 'r') as f:
                analysis = yaml.safe_load(f)
            print(f"\n解析結果 ({repos[0]}):")
            print(yaml.dump(analysis, default_flow_style=False, allow_unicode=True))
        else:
            print(f"\n⚠️ {repos[0]} の解析結果が見つかりません")
else:
    print("まだ解析が実行されていません。")